# 🏥 NESA — Medical RAG System
## Notebook 02 · Semantic Chunking & Contextual Metadata Enrichment

> **Input:**  `data/processed/documents/nesa_documents.jsonl`
> **Output:** `data/processed/chunks/nesa_chunks.jsonl`
>
> **Scope:** Chunking and context injection only.
> Embeddings, vector indexing, retrieval, and generation are out of scope.

---

### Pipeline Overview

```
Cleaned JSONL Blocks
        │
        ▼
 [Group by Document]
        │
        ▼
 [Merge Small Adjacent Blocks]  ← same doc + same section + same content_type
        │
        ▼
 [Atomic Chunks]  ← tables & figure captions always stay whole
        │
        ▼
 [Split Oversized Paragraphs]   ← sliding-window with token-aware overlap
        │
        ▼
 [Inject Contextual Headers]    ← [Document Type] [Section] [Page] prefix
        │
        ▼
 Enriched Chunk JSONL
```


## 1. Configuration & Setup

> All parameters are defined here. Adjust token limits for your embedding model.
> `cl100k_base` is compatible with `text-embedding-3-*` and `gpt-4` family.


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_JSONL  = "data/processed/documents/nesa_documents.jsonl"
OUTPUT_DIR   = "data/processed/chunks"
OUTPUT_JSONL = f"{OUTPUT_DIR}/nesa_chunks.jsonl"

# ── Tokenizer ─────────────────────────────────────────────────────────────────
TIKTOKEN_ENCODING = "cl100k_base"   # compatible with text-embedding-3-*, GPT-4

# ── Chunk size targets ─────────────────────────────────────────────────────────
#
#  Why these values?
#  ─────────────────
#  • MAX_CHUNK_TOKENS = 450  →  leaves headroom under the 512-token limit used by
#    most bi-encoder models (e.g. text-embedding-3-small). Context headers consume
#    ~30-60 tokens, so effective content is ~390-420 tokens.
#
#  • MIN_CHUNK_TOKENS = 50   →  filters out noise fragments that would degrade
#    retrieval precision (single-word headings, page artifacts).
#
#  • CHUNK_OVERLAP_TOKENS = 75  →  ~15-20% overlap ensures sentences at chunk
#    boundaries are represented in at least two chunks, preventing retrieval
#    misses on query-answer splits.
#
MAX_CHUNK_TOKENS     = 450
MIN_CHUNK_TOKENS     = 50
CHUNK_OVERLAP_TOKENS = 75

# ── Document-type routing ──────────────────────────────────────────────────────
#  Postpartum content targets smaller chunks for conversational retrieval.
#  Research/guideline content can be slightly larger for dense synthesis.
POSTPARTUM_DOC_TYPES = {"patient_education"}
RESEARCH_DOC_TYPES   = {"research_paper", "guideline", "book"}

POSTPARTUM_MAX_TOKENS  = 400    # tighter → more self-contained answers
RESEARCH_MAX_TOKENS    = 450    # larger → more context per retrieval hit

# ── Reduction alert ───────────────────────────────────────────────────────────
# Flag any document that loses more than this share of its input tokens
TOKEN_LOSS_ALERT_PCT = 0.02    # 2 % loss triggers a warning

print("✅  Configuration loaded.")
print(f"   Encoding         : {TIKTOKEN_ENCODING}")
print(f"   Max chunk tokens : {MAX_CHUNK_TOKENS}")
print(f"   Min chunk tokens : {MIN_CHUNK_TOKENS}")
print(f"   Overlap tokens   : {CHUNK_OVERLAP_TOKENS}")


## 2. Imports

In [ ]:
import json
import re
import uuid
import warnings
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import tiktoken
import pandas as pd

warnings.filterwarnings("ignore")

# ── Create output directory ───────────────────────────────────────────────────
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ── Initialise tokeniser (global, reused across all calls) ────────────────────
ENCODER = tiktoken.get_encoding(TIKTOKEN_ENCODING)

print(f"✅  tiktoken encoder ready  →  {TIKTOKEN_ENCODING}")


## 3. Data Ingestion

Load the cleaned JSONL produced by Notebook 01. Each line is one content block.


In [ ]:
def load_blocks(path: str) -> List[Dict[str, Any]]:
    """
    Load all content blocks from a JSONL file.
    Raises FileNotFoundError with a helpful message if the file is missing.
    """
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"Input file not found: {path}\n"
            "Run Notebook 01 (Ingestion & Cleaning) first."
        )

    blocks = []
    with open(p, encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                blocks.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"  ⚠️  Skipping malformed line {line_no}: {e}")

    return blocks


# ── Load ──────────────────────────────────────────────────────────────────────
try:
    all_blocks = load_blocks(INPUT_JSONL)
    print(f"✅  Loaded {len(all_blocks):,} blocks from {INPUT_JSONL}")

    # Quick schema preview
    if all_blocks:
        sample = all_blocks[0]
        print("\nSample block keys:", list(sample.keys()))
        print("Sample content_type:", sample.get("content_type"))
        print("Sample text (60 chars):", str(sample.get("text", ""))[:60])

except FileNotFoundError as e:
    print(f"❌  {e}")
    all_blocks = []   # continue notebook execution for demonstration


## 4. Helper Functions

### 4-A · Token Counter

`tiktoken` counts tokens **exactly** as the target embedding model would.
Using `len(text.split())` underestimates by ~20-30 % for medical text
(abbreviations, symbols, formulae each tokenise unexpectedly).


In [ ]:
def count_tokens(text: str) -> int:
    """Return the exact token count for a string using the global encoder."""
    if not text:
        return 0
    return len(ENCODER.encode(text))


def encode_text(text: str) -> List[int]:
    """Return token ids for a string."""
    return ENCODER.encode(text)


def decode_tokens(token_ids: List[int]) -> str:
    """Decode a list of token ids back to a string."""
    return ENCODER.decode(token_ids)


# ── Quick smoke test ──────────────────────────────────────────────────────────
_test = "The patient should take 500 mg of ibuprofen every 6 hours (p < 0.05)."
print(f"Test sentence  : {_test}")
print(f"Token count    : {count_tokens(_test)}")
print(f"Expected range : 16–20 tokens  ({'OK' if 14 <= count_tokens(_test) <= 22 else 'CHECK'})")


### 4-B · Contextual Header Generator

**Why context headers?**

RAG retrieval operates on chunk vectors in isolation. Without a header, the chunk:

> *"Take 500 mg twice daily."*

is meaningless. With a header it becomes:

> *[Document Type: guideline] [Chapter: Pain Management] [Section: Analgesics] [Page: 12]*
> *Take 500 mg twice daily.*

This dramatically improves both dense (embedding cosine) and sparse (BM25 keyword)
retrieval precision for medical queries.


In [ ]:
def build_context_header(block: Dict[str, Any]) -> str:
    """
    Construct a concise contextual header from block metadata.

    Rules:
    - Only include fields that have a real (non-null, non-empty) value.
    - Never inject the string 'null' or 'None'.
    - Keep the header short — headers consume tokens from the chunk budget.
    """
    parts: List[str] = []

    doc_type = block.get("document_type") or ""
    if doc_type:
        parts.append(f"[Document Type: {doc_type}]")

    title = block.get("title") or ""
    if title:
        parts.append(f"[Title: {title}]")

    chapter = block.get("chapter") or ""
    if chapter:
        # Strip markdown heading prefix (# / ## / ###) if present
        chapter_clean = re.sub(r"^#+\s*", "", chapter).strip()
        if chapter_clean:
            parts.append(f"[Chapter: {chapter_clean}]")

    section = block.get("section") or ""
    if section:
        section_clean = re.sub(r"^#+\s*", "", section).strip()
        if section_clean:
            parts.append(f"[Section: {section_clean}]")

    page = block.get("page")
    if page is not None:
        parts.append(f"[Page: {page}]")

    content_type = block.get("content_type") or ""
    if content_type in ("table", "figure_caption"):
        label = "Table" if content_type == "table" else "Figure"
        parts.append(f"[{label}]")

    if not parts:
        return ""

    return " ".join(parts)


# ── Demo ──────────────────────────────────────────────────────────────────────
_demo_block = {
    "document_type": "guideline",
    "title": "WHO Postpartum Care Guidelines",
    "chapter": "# Pain Management",
    "section": "## Analgesics",
    "page": 12,
    "content_type": "paragraph",
}
_demo_header = build_context_header(_demo_block)
print("Demo header:")
print(_demo_header)
print(f"Header tokens: {count_tokens(_demo_header)}")


### 4-C · Block-Level Grouping

Group blocks by `(document_id, document_type)` so the chunker processes
one document at a time and never merges content across document boundaries.


In [ ]:
def group_blocks_by_document(
    blocks: List[Dict[str, Any]]
) -> Dict[str, List[Dict[str, Any]]]:
    """
    Return an ordered dict of document_id → list-of-blocks.
    Blocks within each document are sorted by (page, block_index).
    """
    grouped: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    for block in blocks:
        doc_id = block.get("document_id", "unknown")
        grouped[doc_id].append(block)

    # Sort blocks within each document
    for doc_id in grouped:
        grouped[doc_id].sort(
            key=lambda b: (b.get("page") or 0, b.get("block_index") or 0)
        )

    return dict(grouped)


# ── Demo ──────────────────────────────────────────────────────────────────────
if all_blocks:
    grouped = group_blocks_by_document(all_blocks)
    print(f"Documents found: {len(grouped)}")
    for doc_id, blks in list(grouped.items())[:5]:
        doc_type = blks[0].get("document_type", "?")
        print(f"  {doc_id}  ({doc_type})  →  {len(blks)} blocks")
else:
    grouped = {}
    print("⚠️  No blocks loaded — using empty grouped dict for demo.")


## 5. Chunking Implementation

### 5-A · Sliding-Window Splitter for Oversized Paragraphs

When a single block exceeds `MAX_CHUNK_TOKENS`, we split it token-by-token
using a sliding window. The overlap ensures that sentences spanning a chunk
boundary are retrievable from both adjacent chunks.


In [ ]:
def split_text_with_overlap(
    text: str,
    max_tokens: int,
    overlap_tokens: int,
) -> List[str]:
    """
    Split a long text into overlapping token windows.

    Strategy:
    1. Encode the full text to token ids.
    2. Walk forward by (max_tokens - overlap_tokens) each step.
    3. Decode each window back to a string.

    This guarantees:
    - No window exceeds max_tokens.
    - Consecutive windows share overlap_tokens of context.
    - No token is silently dropped.
    """
    if not text:
        return []

    all_token_ids = encode_text(text)
    total_tokens  = len(all_token_ids)

    if total_tokens <= max_tokens:
        return [text]   # no split needed

    step   = max(1, max_tokens - overlap_tokens)
    chunks = []
    start  = 0

    while start < total_tokens:
        end     = min(start + max_tokens, total_tokens)
        window  = all_token_ids[start:end]
        decoded = decode_tokens(window).strip()
        if decoded:
            chunks.append(decoded)
        if end == total_tokens:
            break
        start += step

    return chunks


# ── Smoke test ────────────────────────────────────────────────────────────────
_long_text = " ".join([
    "The postpartum period requires careful monitoring of the patient's recovery.",
    "Wound care should be performed daily with sterile technique.",
    "Signs of infection include redness, warmth, swelling, and purulent discharge.",
    "Fever above 38.0°C persisting beyond 24 hours postpartum warrants investigation.",
    "Adequate hydration and nutrition are essential for wound healing.",
    "Breastfeeding mothers require an additional 500 kcal per day above baseline.",
    "Iron supplementation is recommended when haemoglobin is below 10 g/dL.",
    "Non-steroidal anti-inflammatory drugs should be used with caution in breastfeeding.",
    "Patient education regarding danger signs must be documented in the clinical record.",
    "Follow-up appointments should be scheduled at 1 week and 6 weeks post-delivery.",
] * 6)   # repeat to make it oversized

_test_chunks = split_text_with_overlap(_long_text, max_tokens=100, overlap_tokens=20)
print(f"Input tokens   : {count_tokens(_long_text)}")
print(f"Chunks produced: {len(_test_chunks)}")
for i, c in enumerate(_test_chunks):
    print(f"  Chunk {i+1}: {count_tokens(c)} tokens | first 60 chars: {c[:60]!r}")


### 5-B · Atomic Chunk Builder (Tables & Figures)

Tables and figure captions are **never split**. Cutting a medication dosing table
in half would produce dangerously incomplete retrieval results.

If a table exceeds `MAX_CHUNK_TOKENS`, we emit it as-is and flag it —
the retrieval system must handle oversized chunks gracefully (e.g., extended context).


In [ ]:
def build_atomic_chunk(
    block: Dict[str, Any],
    chunk_index: int,
    max_tokens: int,
) -> Dict[str, Any]:
    """
    Create a single atomic chunk for a table or figure caption.
    Never splits the content even if it exceeds max_tokens.
    """
    raw_text = (block.get("text") or "").strip()
    header   = build_context_header(block)
    full_text = f"{header}\n\n{raw_text}" if header else raw_text
    tokens   = count_tokens(full_text)

    flags = list(block.get("cleaning_flags") or [])
    if tokens > max_tokens:
        flags.append(f"oversized_atomic_chunk_{tokens}_tokens")

    content_type_map = {
        "table":          "table",
        "figure_caption": "figure",
    }
    out_content_type = content_type_map.get(block.get("content_type", ""), "text")

    doc_id = block.get("document_id", "unknown")

    return {
        "chunk_id":            f"{doc_id}_chunk_{chunk_index:04d}",
        "document_id":         doc_id,
        "source_file":         block.get("source_file"),
        "document_type":       block.get("document_type"),
        "title":               block.get("title") or None,
        "page_start":          block.get("page"),
        "page_end":            block.get("page"),
        "chapter":             block.get("chapter") or None,
        "section":             block.get("section") or None,
        "content_type":        out_content_type,
        "text":                full_text,
        "raw_text":            raw_text,
        "token_count":         tokens,
        "source_block_indices":[block.get("block_index")],
        "cleaning_flags":      flags,
    }

print("✅  build_atomic_chunk() defined.")


### 5-C · Text Block Merger

Adjacent text blocks (paragraphs + headings) that belong to the **same document,
same section, and same page range** are merged until reaching `max_tokens`.

This keeps semantically coherent ideas together — a heading followed by its
explanatory paragraph should live in the same chunk for coherent retrieval.


In [ ]:
# Content types eligible for merging
MERGEABLE_TYPES = {"paragraph", "heading", "reference"}
# Content types that must stay atomic
ATOMIC_TYPES    = {"table", "figure_caption"}


def _same_merge_context(a: Dict[str, Any], b: Dict[str, Any]) -> bool:
    """
    Return True if blocks a and b can be merged.
    Criteria:
    - Same document_id (never cross document boundary).
    - Same section (avoid mixing section content).
    - Both are mergeable content types.
    - Page difference ≤ 1 (allow cross-page paragraph continuation).
    """
    if a.get("document_id") != b.get("document_id"):
        return False
    if a.get("content_type") not in MERGEABLE_TYPES:
        return False
    if b.get("content_type") not in MERGEABLE_TYPES:
        return False
    # Sections must match (None == None is OK — both have no section)
    if (a.get("section") or "") != (b.get("section") or ""):
        return False
    # Allow merging across at most one page boundary
    a_page = a.get("page") or 0
    b_page = b.get("page") or 0
    if abs(a_page - b_page) > 1:
        return False
    return True


def merge_blocks_into_candidates(
    blocks: List[Dict[str, Any]],
    max_tokens: int,
) -> List[List[Dict[str, Any]]]:
    """
    Group adjacent mergeable blocks into candidate merge groups.
    Each group will become one or more chunks.

    Returns a list of groups, where each group is a list of blocks.
    Atomic blocks (tables, figures) each form their own single-block group.
    """
    if not blocks:
        return []

    groups: List[List[Dict[str, Any]]] = []
    current_group: List[Dict[str, Any]] = []
    current_tokens = 0

    for block in blocks:
        b_type = block.get("content_type", "paragraph")
        b_text = (block.get("text") or "").strip()
        b_tokens = count_tokens(b_text)

        # ── Atomic types flush current group and emit alone ───────────────────
        if b_type in ATOMIC_TYPES:
            if current_group:
                groups.append(current_group)
                current_group = []
                current_tokens = 0
            groups.append([block])
            continue

        # ── Skip empty blocks ─────────────────────────────────────────────────
        if b_tokens == 0:
            continue

        # ── Check merge compatibility ─────────────────────────────────────────
        if (current_group
                and _same_merge_context(current_group[-1], block)
                and current_tokens + b_tokens <= max_tokens):
            current_group.append(block)
            current_tokens += b_tokens
        else:
            # Flush current group
            if current_group:
                groups.append(current_group)
            current_group  = [block]
            current_tokens = b_tokens

    if current_group:
        groups.append(current_group)

    return groups

print("✅  merge_blocks_into_candidates() defined.")


### 5-D · Main Chunk Generator

Converts a list of block-groups (from the merger) into final enriched chunks.
Oversized merged groups are split with the sliding-window splitter.


In [ ]:
def _representative_metadata(group: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Extract representative metadata from a group of blocks.
    Takes the first non-null value for each field.
    Page range covers min→max page in the group.
    """
    def first_valid(*vals):
        for v in vals:
            if v is not None and v != "":
                return v
        return None

    pages = [b.get("page") for b in group if b.get("page") is not None]

    return {
        "document_id":   first_valid(*(b.get("document_id")   for b in group)),
        "source_file":   first_valid(*(b.get("source_file")   for b in group)),
        "document_type": first_valid(*(b.get("document_type") for b in group)),
        "title":         first_valid(*(b.get("title")         for b in group)),
        "chapter":       first_valid(*(b.get("chapter")       for b in group)),
        "section":       first_valid(*(b.get("section")       for b in group)),
        "page_start":    min(pages) if pages else None,
        "page_end":      max(pages) if pages else None,
        "block_indices": [b.get("block_index") for b in group],
        "cleaning_flags": [f for b in group for f in (b.get("cleaning_flags") or [])],
    }


def generate_chunks_for_document(
    doc_blocks: List[Dict[str, Any]],
    max_tokens: int,
    min_tokens: int,
    overlap_tokens: int,
) -> List[Dict[str, Any]]:
    """
    Full chunking pipeline for one document's blocks.
    Returns a list of enriched chunk dicts.
    """
    chunks: List[Dict[str, Any]] = []
    chunk_counter = 0

    # ── Step 1: Merge small adjacent blocks ───────────────────────────────────
    groups = merge_blocks_into_candidates(doc_blocks, max_tokens)

    for group in groups:
        b0 = group[0]
        b_type = b0.get("content_type", "paragraph")

        # ── Atomic group (table / figure) ─────────────────────────────────────
        if b_type in ATOMIC_TYPES:
            chunk = build_atomic_chunk(b0, chunk_counter, max_tokens)
            chunks.append(chunk)
            chunk_counter += 1
            continue

        # ── Text group ────────────────────────────────────────────────────────
        meta     = _representative_metadata(group)
        raw_text = "\n\n".join(
            (b.get("text") or "").strip()
            for b in group
            if (b.get("text") or "").strip()
        )

        if not raw_text:
            continue

        # Build header using the representative block (first block)
        header = build_context_header({**b0, **{
            "chapter": meta["chapter"],
            "section": meta["section"],
            "page":    meta["page_start"],
        }})

        # Compute tokens of the content alone (not yet with header)
        content_tokens = count_tokens(raw_text)
        header_tokens  = count_tokens(header)
        budget         = max(1, max_tokens - header_tokens - 2)  # 2 for "\n\n"

        # ── Case 1: fits in one chunk ─────────────────────────────────────────
        if content_tokens <= budget:
            full_text   = f"{header}\n\n{raw_text}" if header else raw_text
            total_tokens = count_tokens(full_text)

            if total_tokens >= min_tokens:
                doc_id = meta["document_id"] or "unknown"
                chunks.append({
                    "chunk_id":            f"{doc_id}_chunk_{chunk_counter:04d}",
                    "document_id":         meta["document_id"],
                    "source_file":         meta["source_file"],
                    "document_type":       meta["document_type"],
                    "title":               meta["title"],
                    "page_start":          meta["page_start"],
                    "page_end":            meta["page_end"],
                    "chapter":             meta["chapter"] or None,
                    "section":             meta["section"] or None,
                    "content_type":        "text",
                    "text":                full_text,
                    "raw_text":            raw_text,
                    "token_count":         total_tokens,
                    "source_block_indices": meta["block_indices"],
                    "cleaning_flags":      meta["cleaning_flags"],
                })
                chunk_counter += 1

        # ── Case 2: too large → sliding-window split ──────────────────────────
        else:
            sub_texts = split_text_with_overlap(raw_text, budget, overlap_tokens)
            for sub_text in sub_texts:
                if not sub_text.strip():
                    continue
                full_text   = f"{header}\n\n{sub_text}" if header else sub_text
                total_tokens = count_tokens(full_text)
                if total_tokens < min_tokens:
                    continue
                doc_id = meta["document_id"] or "unknown"
                chunks.append({
                    "chunk_id":            f"{doc_id}_chunk_{chunk_counter:04d}",
                    "document_id":         meta["document_id"],
                    "source_file":         meta["source_file"],
                    "document_type":       meta["document_type"],
                    "title":               meta["title"],
                    "page_start":          meta["page_start"],
                    "page_end":            meta["page_end"],
                    "chapter":             meta["chapter"] or None,
                    "section":             meta["section"] or None,
                    "content_type":        "text",
                    "text":                full_text,
                    "raw_text":            sub_text,
                    "token_count":         total_tokens,
                    "source_block_indices": meta["block_indices"],
                    "cleaning_flags":      meta["cleaning_flags"] + ["split_chunk"],
                })
                chunk_counter += 1

    return chunks

print("✅  generate_chunks_for_document() defined.")


## 6. Execution Pipeline

Iterate over every document, select the correct token budget based on document type,
run the chunker, and collect all chunks.


In [ ]:
def run_chunking_pipeline(
    grouped: Dict[str, List[Dict[str, Any]]],
) -> List[Dict[str, Any]]:
    """
    Run the full chunking pipeline over all documents.
    Returns a flat list of all enriched chunks.
    """
    all_chunks: List[Dict[str, Any]] = []
    skipped_blocks = 0

    for doc_id, doc_blocks in grouped.items():
        # Determine token budget by document type
        doc_type = (doc_blocks[0].get("document_type") or "").lower()
        if doc_type in POSTPARTUM_DOC_TYPES:
            max_tok = POSTPARTUM_MAX_TOKENS
        else:
            max_tok = RESEARCH_MAX_TOKENS

        chunks = generate_chunks_for_document(
            doc_blocks=doc_blocks,
            max_tokens=max_tok,
            min_tokens=MIN_CHUNK_TOKENS,
            overlap_tokens=CHUNK_OVERLAP_TOKENS,
        )

        all_chunks.extend(chunks)

    return all_chunks, skipped_blocks


# ── Execute ───────────────────────────────────────────────────────────────────
if grouped:
    all_chunks, skipped = run_chunking_pipeline(grouped)
    print(f"✅  Chunking complete.")
    print(f"   Total input blocks : {len(all_blocks):,}")
    print(f"   Total chunks output: {len(all_chunks):,}")
    print(f"   Skipped (too small): {skipped:,}")
else:
    # Demo mode — no real data loaded
    all_chunks = []
    print("⚠️  No blocks loaded. Showing demo mode.")
    print("   Place JSONL output from Notebook 01 at:", INPUT_JSONL)


## 7. Quality Control & Statistics

### 7-A · Token Distribution


In [ ]:
if all_chunks:
    token_counts  = [c["token_count"] for c in all_chunks]
    content_types = [c["content_type"] for c in all_chunks]
    doc_types     = [c.get("document_type") or "unknown" for c in all_chunks]

    df_stats = pd.DataFrame({
        "token_count":  token_counts,
        "content_type": content_types,
        "document_type": doc_types,
    })

    print("=" * 55)
    print("TOKEN DISTRIBUTION — ALL CHUNKS")
    print("=" * 55)
    print(df_stats["token_count"].describe().round(1).to_string())
    print()

    # Distribution buckets
    buckets = {
        f"<  {MIN_CHUNK_TOKENS} (sub-minimum)": (df_stats["token_count"] < MIN_CHUNK_TOKENS).sum(),
        f"{MIN_CHUNK_TOKENS}–200":              ((df_stats["token_count"] >= MIN_CHUNK_TOKENS) & (df_stats["token_count"] < 200)).sum(),
        "200–350":                              ((df_stats["token_count"] >= 200) & (df_stats["token_count"] < 350)).sum(),
        "350–450":                              ((df_stats["token_count"] >= 350) & (df_stats["token_count"] < 450)).sum(),
        f"> {MAX_CHUNK_TOKENS} (oversized)":    (df_stats["token_count"] > MAX_CHUNK_TOKENS).sum(),
    }
    print("Token bucket distribution:")
    for label, count in buckets.items():
        bar = "█" * min(count, 50)
        print(f"  {label:30s}: {count:5,d}  {bar}")

    print()
    print("Chunks by content type:")
    print(df_stats["content_type"].value_counts().to_string())
    print()
    print("Chunks by document type:")
    print(df_stats["document_type"].value_counts().to_string())

else:
    print("No chunks to analyse.")


### 7-B · Coverage Verification (Zero-Loss Check)

In [ ]:
# ── Verify every input block is represented in at least one chunk ─────────────
print("=" * 55)
print("COVERAGE VERIFICATION")
print("=" * 55)

if all_blocks and all_chunks:
    # Collect all block indices referenced by chunks
    covered_indices = set()
    for chunk in all_chunks:
        for idx in (chunk.get("source_block_indices") or []):
            if idx is not None:
                covered_indices.add(idx)

    all_block_indices = set(
        b.get("block_index")
        for b in all_blocks
        if b.get("block_index") is not None
        and (b.get("text") or "").strip()               # non-empty blocks only
        and count_tokens((b.get("text") or "").strip()) >= MIN_CHUNK_TOKENS
    )

    uncovered = all_block_indices - covered_indices
    coverage_pct = 100 * (1 - len(uncovered) / max(len(all_block_indices), 1))

    print(f"  Non-empty blocks above min token threshold : {len(all_block_indices):,}")
    print(f"  Blocks referenced in at least one chunk    : {len(covered_indices):,}")
    print(f"  Uncovered blocks                           : {len(uncovered):,}")
    print(f"  Coverage                                   : {coverage_pct:.1f} %")

    if uncovered:
        print("\n  ⚠️  Uncovered block indices (first 10):", sorted(uncovered)[:10])
        print("     Inspect these blocks — they may be sub-minimum or empty.")
    else:
        print("\n  ✅  All qualifying blocks are covered.")

    # Oversized atomic chunks
    oversized = [c for c in all_chunks if f"oversized_atomic_chunk" in " ".join(c.get("cleaning_flags") or [])]
    if oversized:
        print(f"\n  ⚠️  Oversized atomic chunks (tables/figures > MAX_CHUNK_TOKENS): {len(oversized)}")
        for c in oversized[:3]:
            print(f"     {c['chunk_id']}  {c['token_count']} tokens")
    else:
        print("  ✅  No oversized atomic chunks.")

else:
    print("  ⚠️  Skipped — no blocks or chunks loaded.")


### 7-C · Per-Document Statistics

In [ ]:
if all_chunks:
    # Aggregate per document
    doc_stats = defaultdict(lambda: {"chunk_count": 0, "total_tokens": 0, "doc_type": ""})
    for c in all_chunks:
        d = c.get("document_id", "?")
        doc_stats[d]["chunk_count"]  += 1
        doc_stats[d]["total_tokens"] += c.get("token_count", 0)
        doc_stats[d]["doc_type"]      = c.get("document_type", "?")

    df_docs = pd.DataFrame([
        {
            "document_id":  doc_id,
            "doc_type":     v["doc_type"],
            "chunks":       v["chunk_count"],
            "total_tokens": v["total_tokens"],
            "avg_tokens":   round(v["total_tokens"] / max(v["chunk_count"], 1), 1),
        }
        for doc_id, v in doc_stats.items()
    ]).sort_values("chunks", ascending=False)

    print("Per-document chunk statistics:")
    display(df_docs)
else:
    print("No chunks to display.")


## 8. Output Generation

Save all enriched chunks to `data/processed/chunks/nesa_chunks.jsonl`.
Each line is one self-contained JSON object, ready for the embedding notebook.


In [ ]:
def save_chunks_jsonl(chunks: List[Dict[str, Any]], output_path: str) -> None:
    """Write chunks to a JSONL file (one JSON object per line, UTF-8)."""
    out = Path(output_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8") as f:
        for chunk in chunks:
            f.write(json.dumps(chunk, ensure_ascii=False) + "\n")
    print(f"✅  Saved {len(chunks):,} chunks → {output_path}")


if all_chunks:
    save_chunks_jsonl(all_chunks, OUTPUT_JSONL)

    # ── Verify file integrity ─────────────────────────────────────────────────
    with open(OUTPUT_JSONL, encoding="utf-8") as f:
        saved_lines = sum(1 for line in f if line.strip())

    assert saved_lines == len(all_chunks), (
        f"Line count mismatch: saved {saved_lines}, expected {len(all_chunks)}"
    )
    file_size_kb = Path(OUTPUT_JSONL).stat().st_size // 1024
    print(f"   Lines verified   : {saved_lines:,}")
    print(f"   File size        : {file_size_kb:,} KB")
else:
    print("⚠️  No chunks to save — run pipeline first.")


## 9. Data Inspection

### 9-A · Sample Chunks — Postpartum (Patient Education)


In [ ]:
def print_chunk(chunk: Dict[str, Any], label: str = "") -> None:
    """Pretty-print a single chunk for inspection."""
    sep = "─" * 65
    print(sep)
    if label:
        print(f"  {label}")
    print(f"  chunk_id       : {chunk.get('chunk_id')}")
    print(f"  document_type  : {chunk.get('document_type')}")
    print(f"  content_type   : {chunk.get('content_type')}")
    print(f"  pages          : {chunk.get('page_start')} → {chunk.get('page_end')}")
    print(f"  section        : {chunk.get('section') or '—'}")
    print(f"  token_count    : {chunk.get('token_count')}")
    print(f"  flags          : {chunk.get('cleaning_flags') or 'none'}")
    print(f"  source_blocks  : {chunk.get('source_block_indices')}")
    print()
    print("  ── FULL TEXT (with context header) ──")
    text = chunk.get("text", "")
    print("  " + text[:800].replace("\n", "\n  "))
    if len(text) > 800:
        print("  ... [truncated]")
    print(sep)


# Show up to 3 postpartum chunks
postpartum_chunks = [
    c for c in all_chunks
    if (c.get("document_type") or "") in POSTPARTUM_DOC_TYPES
]

if postpartum_chunks:
    print(f"Showing {min(3, len(postpartum_chunks))} postpartum chunk(s):\n")
    for c in postpartum_chunks[:3]:
        print_chunk(c, label="Postpartum / Patient Education")
else:
    print("No postpartum chunks found.")
    if all_chunks:
        print("(Showing first available chunk instead)\n")
        print_chunk(all_chunks[0], label="First available chunk")


### 9-B · Sample Chunks — Research / Clinical Guidelines

In [ ]:
research_chunks = [
    c for c in all_chunks
    if (c.get("document_type") or "") in RESEARCH_DOC_TYPES
]

if research_chunks:
    print(f"Showing {min(3, len(research_chunks))} research chunk(s):\n")
    for c in research_chunks[:3]:
        print_chunk(c, label="Research / Guideline")
else:
    print("No research/guideline chunks found.")


### 9-C · Sample Table Chunk

In [ ]:
table_chunks = [c for c in all_chunks if c.get("content_type") == "table"]

if table_chunks:
    print(f"Total table chunks: {len(table_chunks)}")
    print_chunk(table_chunks[0], label="Table (Atomic Chunk)")
else:
    print("No table chunks found in current dataset.")

    # ── Demo: show what a table chunk looks like ──────────────────────────────
    print("\n[DEMO] Synthetic table chunk example:\n")
    demo_table_block = {
        "document_id":    "demo_doc_001",
        "source_file":    "data/raw/guidelines/demo.pdf",
        "document_type":  "guideline",
        "title":          "WHO Postpartum Guidelines",
        "chapter":        "# Medications",
        "section":        "## Analgesics",
        "page":           14,
        "content_type":   "table",
        "block_index":    42,
        "cleaning_flags": [],
        "text": (
            "TABLE (page 14):\n"
            "Columns: Medication | Dose | Frequency | Route\n"
            "- Medication = Ibuprofen; Dose = 400 mg; Frequency = every 6 hours; Route = oral.\n"
            "- Medication = Acetaminophen; Dose = 500 mg; Frequency = every 4-6 hours; Route = oral.\n"
            "- Medication = Oxytocin; Dose = 10 IU; Frequency = as needed; Route = IV / IM."
        ),
    }
    demo_chunk = build_atomic_chunk(demo_table_block, chunk_index=0, max_tokens=MAX_CHUNK_TOKENS)
    print_chunk(demo_chunk, label="Table (Atomic Chunk — Demo)")


### 9-D · Sample Split Chunk (Sliding Window)

In [ ]:
split_chunks = [
    c for c in all_chunks
    if "split_chunk" in (c.get("cleaning_flags") or [])
]

print(f"Chunks produced by sliding-window split: {len(split_chunks)}")

if split_chunks:
    print_chunk(split_chunks[0], label="Split Chunk (from oversized paragraph)")
else:
    print("No split chunks in current dataset — all paragraphs fit within budget.")


## 10. Final Summary & Next Steps

In [ ]:
print("=" * 55)
print("NESA — NOTEBOOK 02 SUMMARY")
print("=" * 55)

if all_chunks:
    token_counts = [c["token_count"] for c in all_chunks]
    print(f"  Input blocks        : {len(all_blocks):,}")
    print(f"  Output chunks       : {len(all_chunks):,}")
    print(f"  Average chunk size  : {sum(token_counts)/len(token_counts):.0f} tokens")
    print(f"  Min chunk size      : {min(token_counts)} tokens")
    print(f"  Max chunk size      : {max(token_counts)} tokens")
    print(f"  Table chunks        : {sum(1 for c in all_chunks if c['content_type'] == 'table')}")
    print(f"  Figure chunks       : {sum(1 for c in all_chunks if c['content_type'] == 'figure')}")
    print(f"  Split chunks        : {sum(1 for c in all_chunks if 'split_chunk' in (c.get('cleaning_flags') or []))}")
    print(f"  Output file         : {OUTPUT_JSONL}")
else:
    print("  ⚠️  Pipeline ran in demo mode — no real data processed.")

print()
print("📌  Next steps:")
print("   Notebook 03 → Embedding Generation")
print("             (Use chunk['text'] as input to text-embedding-3-small or similar.)")
print("   Notebook 04 → Vector DB Indexing (Qdrant / Weaviate / pgvector)")
print("   Notebook 05 → Hybrid Retrieval  (dense + BM25 + reranker)")
print("   Notebook 06 → RAG Generation    (Postpartum & Research answer synthesis)")
